In [417]:
from sklearn import preprocessing
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score, mean_absolute_percentage_error


# For 2D analysis
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit
from sklearn.preprocessing import MinMaxScaler
# from utils import period2freq, freq2period

# For PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import csv
import time
import glob

# Datasize in KB
data_size_kb = {'4mb': 4096, '16mb': 16384, '64mb': 65536,
            '256mb': 262144, '512mb': 524288, '1gb': 1048576,
            '5gb': 5242880, '50gb': 52428800, '100gb': 104857600,
            '300gb': 314572800,}

# Key Parameters
IOR_PARAMS = ['operation', 'randomOffset', 'transferSize', 
            'aggregateFilesizeMB', 'numTasks', 'totalTime', 
            'numNodes', 'tasksPerNode', 'bwMiB', "storageType"]

TARGET_PARAMS = [ "bestStorage" ]
op_dict = {0: "write", 1: "read"}


## Bandwidth Prediction from IOR to Workflow
### Key parameters from IOR
Parameters for emulation:
- D: Total I/O size per task per operation (take log scale)
- C: Total I/O operation count per operation type, [Cr, Cw]
- A: Average I/O size per operation type, [Ar, Aw]
- IOT: I/O opeartion type per task (seq/ran, read/write), [OTsr, OTsw, OTrr, OTrw]
- N: Number of parallelism/tasks 
- OSC: Size of OS buffered cache
- Mem: Size of memory
Parameters for prediciton:
- TIO: Total I/O size for all tasks
- T: Time spend on the type of I/O operation per task, [Tr, Tw]
- BW: Bandwidth calculated from TIO/T
### Special note for IOR
IOR calculates bwMiB, but the bwMiB is little slower than the (aggregate filesize)/totalTime, but not much difference, particularly in the `totalTime=openTimewrRdTime+closetime`

### Key parameters from workflow
Parameters for characteristics
- D: Total I/O size per task per operation (take log scale)
- C: Total I/O operation count per operation type, [Cr, Cw]
- A: Average I/O size per operation type, calculate from A=D/C [Ar, Aw]
- IOT: I/O opeartion type per task (seq/ran, read/write), [OTsr, OTsw, OTrr, OTrw]
- N: Number of parallelism/tasks 
- OSC: Size of OS buffered cache
- Mem: Size of memory
Parameters for prediciton:
- TIO: Total I/O size for all tasks
- T: Time spend on the type of I/O operation per task, [Tr, Tw]
- BW: Bandwidth calculated from TIO/T

### Perf Modeling Parameters
- a: Performance change when total data size reaches memory buffer cache of system.
- b: Performance change when  total data size reaches memory size of system. (Constant value)
- v: Performance variation due to shared storage resource. 
(The peak variation in the system. Measure with IOR.)
Deception BeeGFS: run at off hour to keep below 20% variation.
Options: (1) User/system admin to ensure the variation percentage; 
(2) Obtain variation with IOR emulation (possible over 20%)
- r: Range of expected performance: Make correct choice with given storage options.
- LPMR: Layered performance matching number for comparing application storage choice.
- w: Weight of a application LPMR. (Total data size, cost of moving data)



# TODO
- normalize BW differently! use same log scale for all (DONE)
- BeeGFS graph show variance:
    - identify whether the performance changes is due to BeeGFS parameters or due to traffic
    - parameter: distribution, tell us the load at particular time, background traffic under a percentage 5%, parameter it self can be a range.
    - Microbenchmark on background load, in principle that we know this exists and we assume we know a load factor, adjust the results. Whether it affect the storage choice for a type of workload.
    

In [418]:
def byte_size_to_human_size(byte_size):
    if byte_size < 1024:
        return f"{byte_size} B"
    elif byte_size < 1024**2:
        return f"{byte_size/1024} KiB"
    elif byte_size < 1024**3:
        return f"{byte_size/1024**2} MiB"
    elif byte_size < 1024**4:
        return f"{byte_size/1024**3} GiB"
    else:
        return f"{byte_size/1024**4} TiB"

# def file_size_to_mb(file_size):
#     # If file_size is type string, convert to float
#     if type(file_size) == str:
#         # Translate KiB, MiB, and GiB to bytes
#         size_num = float(file_size.split(" ")[0])
#         size_unit = file_size.split(" ")[1]
#         if size_unit == "KiB":
#             return float(size_num/1024)
#         elif size_unit == "MiB":
#             return float(size_num)
#         elif size_unit == "GiB":
#             return float(size_num * 1024)
#         else:
#             # If file_size is type int, convert to MB
#             return int(size_num/1024**2)
#     else:
#         # If file_size is type int, convert to MB
#         return int(file_size/1024**2)

def file_size_to_mb(file_size):
    # If file_size is a string, convert to float
    if isinstance(file_size, str):
        # Translate KiB, MiB, and GiB to bytes
        size_num, size_unit = file_size.split()
        size_num = float(size_num)
        
        if size_unit == "KiB":
            return size_num / 1024  # Convert KiB to MB
        elif size_unit == "MiB":
            return size_num  # Already in MB
        elif size_unit == "GiB":
            return size_num * 1024  # Convert GiB to MB
        else:
            raise ValueError(f"Unknown size unit: {size_unit}")
    elif isinstance(file_size, (int, float)):
        # If file_size is an integer or float, assume it's in bytes and convert to MB
        return file_size / (1024 ** 2)  # Convert bytes to MB
    else:
        raise TypeError("file_size must be a string or a number")

def transform_store_code(storage_type):
    if storage_type == "localssd":
        store_code = 0
    elif storage_type == "beegfs":
        store_code = 1
    elif storage_type == "lustre":
        store_code = 2
    else:
        store_code = 3
    return store_code

def decode_store_code(store_code):
    if store_code == 0:
        storage_type = "localssd"
    elif store_code == 1:
        storage_type = "beegfs"
    elif store_code == 2:
        storage_type = "lustre"
    else:
        storage_type = "unknown"
    return storage_type

def get_ior_json_df(ior_test_json_files, storage_type, IOR_PARAMS):


    # Create a dataframe for storing ior parameters
    ior_df = pd.DataFrame(columns=IOR_PARAMS)
    store_code = transform_store_code(storage_type)

    # Load Json files
    for jfile in ior_test_json_files:
        with open(jfile) as f:
            try:
                data = json.load(f)
            except:
                print("Error reading file: ", jfile)
                continue
            # print(jfile)
            
            # Write then Read
            ior_options = data['tests'][0]["Options"]
            IOR_PARAMS = data['tests'][0]["Parameters"]

            # Add write statistics first
            if file_size_to_mb(ior_options['aggregate filesize']) != 0:
                tmp_write_stat = {}
                ior_write_summary = data['summary'][0]
                tmp_write_stat['randomOffset'] = IOR_PARAMS['randomOffset']
                tmp_write_stat['aggregateFilesizeMB'] = file_size_to_mb(ior_options['aggregate filesize'])
                tmp_write_stat['numTasks'] = ior_write_summary['numTasks']
                tmp_write_stat['tasksPerNode'] = ior_write_summary['tasksPerNode']
                tmp_write_stat['numNodes'] = tmp_write_stat['numTasks']/tmp_write_stat['tasksPerNode']
                tmp_write_stat['transferSize'] = ior_write_summary['transferSize']
                if ior_write_summary['operation'] == "write":
                    tmp_write_stat['operation'] = 0
                else:
                    tmp_write_stat['operation'] = -1
                tmp_write_stat['totalTime'] = ior_write_summary['MeanTime'] # only 1 operation per test
                tmp_write_stat['bwMiB'] = ior_write_summary['bwMeanMIB'] # only 1 operation per test
                tmp_write_stat['storageType'] = store_code
                # Add tmp_write_stat to df
                ior_df = ior_df._append(tmp_write_stat, ignore_index=True)

                # Add read statistics
                tmp_read_stat = {}
                ior_read_summary = data['summary'][1]
                tmp_read_stat['randomOffset'] = IOR_PARAMS['randomOffset']
                tmp_read_stat['aggregateFilesizeMB'] = file_size_to_mb(ior_options['aggregate filesize'])
                tmp_read_stat['numTasks'] = ior_read_summary['numTasks']
                tmp_read_stat['tasksPerNode'] = ior_read_summary['tasksPerNode']
                tmp_read_stat['numNodes'] = tmp_read_stat['numTasks']/tmp_read_stat['tasksPerNode']
                tmp_read_stat['transferSize'] = ior_read_summary['transferSize']
                if ior_read_summary['operation'] == "read":
                    tmp_read_stat['operation'] = 1
                else:
                    tmp_read_stat['operation'] = -1
                tmp_read_stat['totalTime'] = ior_read_summary['MeanTime']
                tmp_read_stat['bwMiB'] = ior_read_summary['bwMeanMIB']
                tmp_read_stat['storageType'] = store_code
                # Add tmp_read_stat to df
                ior_df = ior_df._append(tmp_read_stat, ignore_index=True)

    # encode string to numbers
    le = preprocessing.LabelEncoder()
    for col in ior_df.columns:
        if pd.api.types.is_string_dtype(ior_df[col]):
            ior_df[col] = le.fit_transform(ior_df[col])

    # print(ior_df.head(5))

    return ior_df

def plot_heatmap(corrM,outfile):
    plt.figure(figsize=(14, 8))
    #labels = list(corrM.columns)
    plt.subplots_adjust(bottom=0.19)

    # # use only lower triangle
    # corrM = corrM.where(np.triu(np.ones(corrM.shape)).astype(np.bool))

    ## plot all correlation heatmap
    map = sns.heatmap(corrM, vmin=-1, vmax=1,
        linewidths=0.5, linecolor='grey', cmap='BrBG') #annot=True,
    map.set_title('Correlation Matrix Heatmap', fontdict={'fontsize':12}, pad=12)
    #plt.show()
    out_file=f'{outfile}.png'
    plt.savefig(out_file)
    plt.clf()

def get_redundant_pairs(df):
    '''Get diagonal and lower triangular pairs of correlation matrix'''
    pairs_to_drop = set()
    cols = df.columns
    for i in range(0, df.shape[1]):
        for j in range(0, i+1):
            pairs_to_drop.add((cols[i], cols[j]))
    return pairs_to_drop
    
def corrM_sorted_csv(corrM, outfile):
  n=5
  au_corr = corrM.corr().abs().unstack()
  labels_to_drop = get_redundant_pairs(corrM)
  au_corr = au_corr.drop(labels=labels_to_drop).sort_values(ascending=False)
  sorted_corrM = au_corr[0:n]

  out_file= outfile
  sorted_corrM.to_csv(out_file)

def corr_matrix(df,outname=""):

    # calculte correlation matrix
    corrM = df.corr()

    corrM.to_csv(f'{outname}.csv')
    plot_heatmap(corrM,outname)
    #corrM_sorted_csv(corrM, f'sorted_{outname}.csv')

def _2d_trend(X, y, x_label="x-axis", y_label="bwMiB", title="", show=False):
        # X = X.reshape(-1, 1)
        X = X.values.reshape(-1, 1)
        # poly = PolynomialFeatures(degree=2)
        poly = PolynomialFeatures(degree=1)
        poly_data = poly.fit_transform(X)
        model = LinearRegression()
        model.fit(poly_data,y)
        coef = model.coef_
        intercept = model.intercept_
        # Set figure size (80,50)
        plt.figure(figsize=(10,5))

        plt.scatter(X,y,color='red')
        plt.plot(X,model.predict(poly.fit_transform(X)),color='blue')
        # Show the fit parameters on graph with scientific notation
        plt.annotate(f"y = {coef[1]:.2e}x + {intercept:.2e}", xy=(0.05, 0.95), xycoords='axes fraction')

        # print(f"y = {coef[1]}x + {intercept}")
        plt.legend(['Original','Prediction'])
        plt.xlabel(x_label)
        plt.ylabel(y_label)
        plt.title(title)
        if show:
            plt.show()
        return coef[1], intercept

def _3d_trend(x,y,z, title=""):
    # ref: https://stackoverflow.com/questions/2298390/fitting-a-line-in-3d
    # Convert Pandas Series to NumPy arrays
    x = x.to_numpy()
    y = y.to_numpy()
    z = z.to_numpy()
    
    data = np.concatenate((x[:, np.newaxis], 
                        y[:, np.newaxis], 
                        z[:, np.newaxis]), 
                        axis=1)
    
    data = data.astype('float64')

    # Calculate the mean of the points, i.e. the 'center' of the cloud
    datamean = data.mean(axis=0)
    print(datamean)

    # Do an SVD on the mean-centered data.
    # full_matrices=False reduce memory
    uu, dd, vv = np.linalg.svd(data - datamean, full_matrices=False)

    # Get the sptread of data with mean 0 from all axis
    x_min = np.min(data[:,0])
    y_min = np.min(data[:,1])
    z_min = np.min(data[:,2])

    x_max = np.max(data[:,0])
    y_max = np.max(data[:,1])
    z_max = np.max(data[:,2])

    low_bound = min(x_min, y_min, z_min)
    high_bound = max(x_max, y_max, z_max)

    # Now vv[0] contains the first principal component, i.e. the direction
    # vector of the 'best fit' line in the least squares sense.
    # Adjust axist limits (Optional)
    linepts = vv[0] * np.mgrid[low_bound:high_bound:2j][:, np.newaxis]
    # shift by the mean to get the line in the right place
    linepts += datamean

    # Verify that everything looks right.

    # import mpl_toolkits.mplot3d as m3d
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter3D(*data.T)
    ax.plot3D(*linepts.T)
    # ax.scatter3D(data[:,0], data[:,1], data[:,2])
    ax.set_xlabel("transferSize")
    ax.set_ylabel("aggregateFilesizeMB")
    ax.set_zlabel("bwMiB")
    ax.set_title(title)
    plt.show()



In [419]:

def _polynomial_fit_pca(df, title):
    # Standardizing the data
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df)

    # Performing PCA
    pca = PCA(n_components=2)  # Reduce to 2 components for visualization
    principal_components = pca.fit_transform(scaled_data)

    # Creating a DataFrame with the principal components
    pca_df = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])

    # Define a polynomial function to fit
    def polynomial_func(x, a, b, c):
        return a * x**2 + b * x + c

    # Fit the polynomial curve
    params, _ = curve_fit(polynomial_func, pca_df['PC1'], pca_df['PC2'])

    # Predict values using the fitted curve
    x_vals = np.linspace(pca_df['PC1'].min(), pca_df['PC1'].max(), 100)
    fitted_curve = polynomial_func(x_vals, *params)

    # Visualizing the PCA result and the fitted curve
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x='PC1', y='PC2', data=pca_df, label='PCA Data')
    plt.plot(x_vals, fitted_curve, color='red', label='Polynomial Fit')
    plt.title(title)
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend()
    plt.show()

    # Curve fit parameters
    print(f'Polynomial fit parameters: a={params[0]}, b={params[1]}, c={params[2]}')

    # Reconstructing the data from the principal components
    reconstructed_data = pca.inverse_transform(principal_components)

    # Calculating the reconstruction error
    mse = mean_squared_error(scaled_data, reconstructed_data)
    print(f'Reconstruction error (MSE): {mse}')

    # Explained variance
    explained_variance = pca.explained_variance_ratio_
    print(f'Explained variance by principal components: {explained_variance}')


def _linear_fit_pca(df, title):
    # Standardizing the data
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df)

    # Performing PCA
    pca = PCA(n_components=2)  # Reduce to 2 components for visualization
    principal_components = pca.fit_transform(scaled_data)

    # Creating a DataFrame with the principal components
    pca_df = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])

    # Fit the linear regression model
    reg = LinearRegression()
    reg.fit(pca_df[['PC1']], pca_df['PC2'])

    # Predict values using the fitted model
    x_vals = np.linspace(pca_df['PC1'].min(), pca_df['PC1'].max(), 100)
    fitted_line = reg.predict(x_vals.reshape(-1, 1))

    # Visualizing the PCA result and the fitted line
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x='PC1', y='PC2', data=pca_df, label='PCA Data')
    plt.plot(x_vals, fitted_line, color='red', label='Linear Fit')
    plt.title(title)
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend()
    plt.show()

    # Linear fit parameters
    print(f'Linear fit parameters: intercept={reg.intercept_}, slope={reg.coef_[0]}')

    # Reconstructing the data from the principal components
    reconstructed_data = pca.inverse_transform(principal_components)

    # Calculating the reconstruction error
    mse = mean_squared_error(scaled_data, reconstructed_data)
    print(f'Reconstruction error (MSE): {mse}')

    # Explained variance
    explained_variance = pca.explained_variance_ratio_
    print(f'Explained variance by principal components: {explained_variance}')


def ior_json_to_df(ior_data_path):
    # Get the list of JSON files from the directory
    ior_test_json_files = glob.glob(ior_data_path + "/*.json")

    # Get storage type from data_path name
    # storage_type =  ior_data_path.split("/")[-1].split("_")[0] + "_" + ior_data_path.split("/")[-1].split("_")[1]
    storage_type =  ior_data_path.split("/")[-1].split("_")[0]
    # print("Storage Type: ", storage_type)
    ior_df = get_ior_json_df(ior_test_json_files, storage_type, IOR_PARAMS)

    # corr_matrix(ior_df, storage_type)

    return ior_df

# Take a column name for y_data
def x_mean_std(group, y_column):
    return pd.Series({
        f'{y_column}_ave': group[y_column].mean(),
        f'{y_column}_ave_std': group[y_column].std(ddof=0)  # Use population standard deviation by setting ddof=0
    })


def my_data_transform(df, cols_to_norm=[], cols_to_log=[]):
    # Create an empty DataFrame to store the results
    results = []

    # Get unique transfer sizes and number of tasks
    xfersizes = sorted(df['transferSize'].unique())
    numTasks = sorted(df['numTasks'].unique())
    op_type = [0, 1]  # Define outside the loop to avoid redefinition
    print("Transfer Sizes: ", xfersizes)
    print("Number of Tasks: ", numTasks)

    for xfer in xfersizes:
        for t in numTasks:
            for op in op_type:
                # Filter the DataFrame for specific conditions
                subdf = df[(df['numTasks'] == t) &
                           (df['transferSize'] == xfer) &
                           (df['operation'] == op)].copy()
                
                if not subdf.empty:
                    # Calculate the mean and standard deviation of the bwMiB column
                    ave_bw_df = subdf.groupby('aggregateFilesizeMB').apply(
                        lambda group: x_mean_std(group, 'bwMiB')
                    ).reset_index()

                    # Calculate percentage of standard deviation
                    ave_bw_df['bwMiB_ave_std_perc'] = (ave_bw_df['bwMiB_ave_std'] /
                                                       ave_bw_df['bwMiB_ave']) * 100

                    # Merge the aggregated results with the original DataFrame
                    merged_df = pd.merge(subdf, ave_bw_df, on='aggregateFilesizeMB', how='left')

                    # Append the merged DataFrame to the results list
                    results.append(merged_df)

    # Concatenate all the DataFrames in the results list
    new_df = pd.concat(results, ignore_index=True)

    # Log transformation
    if cols_to_log:
        log_new_cols = [f"{col}_log" for col in cols_to_log]
        print("Columns to log: ", cols_to_log)
        new_df[log_new_cols] = np.log(new_df[cols_to_log])

    # Min-Max normalization
    if cols_to_norm:
        norm_new_cols = [f"{col}_norm" for col in cols_to_norm]
        print("Columns to normalize: ", cols_to_norm)
        scaler = MinMaxScaler()
        new_df[norm_new_cols] = scaler.fit_transform(new_df[cols_to_norm])

    return new_df

def oscillatory_func(x, amplitude, frequency, phase, offset):
    return amplitude * np.sin(frequency * x + phase) + offset

def damped_sine_func(x, amplitude, frequency, phase, offset, decay):
    return amplitude * np.sin(frequency * x + phase) * np.exp(-decay * x) + offset

def cos_func(x, amplitude, frequency):
    return amplitude * np.cos(frequency * x)

def normalize_y(y_data):
    scaler = MinMaxScaler(feature_range=(-1, 1))
    return scaler.fit_transform(y_data.values.reshape(-1, 1)).flatten()

def _oscillatory_trend(x_data, y_data, x_label="x-axis", y_label="bwMiB", title="", 
                       show=False, x_datalabel=[], y_datalabel=[], save_path=""):
    # Normalize y_data
    y_data_normalized = y_data #normalize_y(y_data)
    
    # Initial guess for the parameters
    amplitude_guess = (np.max(y_data_normalized) - np.min(y_data_normalized)) / 2
    frequency_guess = 10
    if y_data_normalized[1] > y_data_normalized[0]:
        phase_guess = 0
    else:
        phase_guess = np.pi
    offset_guess = np.mean(y_data_normalized)
    initial_guess = [amplitude_guess, frequency_guess, phase_guess, offset_guess]

    try:
        # Fit the oscillatory function to the normalized data
        params, params_covariance = curve_fit(oscillatory_func, x_data, y_data_normalized, p0=initial_guess, maxfev=10000)
        # params, params_covariance = curve_fit(cos_func, x_data, y_data_normalized, p0=(amplitude_guess, frequency_guess), maxfev=10000)
    except RuntimeError as e:
        print(f"Sine wave fitting failed: {e}")
        # Attempt to fit a damped sine wave if the initial fit fails
        decay_guess = 0.1
        frequency_guess = 20
        # phase_guess2 = 1
        phase_guess = np.pi
        updated_guess = [amplitude_guess, frequency_guess, phase_guess, offset_guess]
        try:
            params, params_covariance = curve_fit(oscillatory_func, x_data, y_data_normalized, p0=initial_guess, maxfev=10000)
        except RuntimeError as e:
            print(f"Sine wave re-fitting failed: {e}")
            return

    # Print the fitted parameters
    print("Fitted parameters:", params)

    # Generate y values based on the fitted parameters
    if len(params) == 4:
        y_fitted = oscillatory_func(x_data, *params)
        equation = f"y = {params[0]:.2f} * sin({params[1]:.2e}x + {params[2]:.2e}) + {params[3]:.2e}"
    else:
        y_fitted = damped_sine_func(x_data, *params)
        equation = f"y = {params[0]:.2f} * sin({params[1]:.2e}x + {params[2]:.2e}) * exp(-{params[4]:.2e}x) + {params[3]:.2e}"

    # set figure size
    plt.figure(figsize=(10,5))

    # Plot the original data as a scatter plot
    plt.scatter(x_data, y_data_normalized, label='Data')

    # Plot the fitted oscillatory curve
    plt.plot(x_data, y_fitted, color='red', label='Fitted Curve')

    # Add data labels foreach point
    x_label_list = []
    y_label_list = []
    if len(x_datalabel) == len(x_data):
        for i, label in enumerate(x_datalabel):
            x_label_list.append(label)
    if len(y_datalabel) == len(y_data):
        for i, label in enumerate(y_datalabel):
            y_label_list.append(label)
    for i in range(len(x_data)):
        plt.text(x_data[i], y_data_normalized[i], f"{x_label_list[i]}, {y_label_list[i]}", fontsize=8, ha='left', va='bottom')
    
    # Always plot x and y from 0 to 1
    plt.xlim(0, 1)
    # plt.ylim(0, 1)

    # Add labels and legend
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    if title:
        plt.title(title)
    else:
        plt.title('Scatter Plot with Curve Fit')
    plt.legend()
    
    # Write the equation of the fitted curve on the plot
    plt.annotate(equation, xy=(0.05, 0.25), xycoords='axes fraction')
    # Add phase_guess and frequency_guess to the plot
    plt.annotate(f"Phase: {params[2]:.2e}, Frequency: {params[1]:.2e}", xy=(0.05, 0.20), xycoords='axes fraction')

    if show:
        # Show the plot
        plt.show()
    
    if save_path != "":
        # save with plt title
        plt.savefig(f'{save_path}/{title}.png')



In [420]:
def merge_by_test_param(ssd_df, beegfs_df):
    # Compare and find the row with only storage_type and bwMiB columns different but others are the same
    # Merge the dataframes on columns other than 'storageType' and 'bwMiB'
    merge_columns = ['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMB', 'numTasks', 'numNodes', 'tasksPerNode', 'opCount'] # 'totalTime', 
    merged_df = pd.merge(ssd_df, beegfs_df, on=merge_columns, suffixes=('_ssd', '_beegfs'), how='outer')

    # Remove columns storageType_ssd and storageType_beegfs
    merged_df.drop(['storageType_ssd', 'storageType_beegfs'], axis=1, inplace=True)
    
    # # Check if cloumens 'operation_ssd' and 'operation_beegfs' has the same values, if yes combine to 1 column 'operation'
    # merged_df['operation'] = merged_df.apply(lambda row: 0 if row['operation_ssd'] == row['operation_beegfs'] else -1, axis=1)

    # Compare 'bwMiB' values and add 'selectStorage' column
    merged_df['selectStorage'] = merged_df.apply(lambda row: 0 if row['bwMiB_ssd'] > row['bwMiB_beegfs'] else 1, axis=1)

    # Debug: Print unique values of 'operation' column before and after merge
    print("Unique 'operation' values in ssd_df:", ssd_df['operation'].unique())
    print("Unique 'operation' values in beegfs_df:", beegfs_df['operation'].unique())
    print("Unique 'operation' values in merged_df:", merged_df['operation'].unique())
    
    print(merged_df.head(5))
    # print shaoe
    print(f"merged_df.shape: {merged_df.shape}")
    print(f"ssd_df.shape: {ssd_df.shape}")
    print(f"beegfs_df.shape: {beegfs_df.shape}")

    return merged_df




In [421]:
def is_sequential(numbers):
    if not numbers:  # Check if the list is empty
        return False

    sorted_numbers = sorted(numbers)  # Sort the numbers
    return all(sorted_numbers[i] + 1 == sorted_numbers[i + 1] for i in range(len(sorted_numbers) - 1))


# FIXME: use folder name as task label, do not use taskName and taskPID
def get_wf_result_df(tests, wf_params, target_tasks, 
                     numTasksWrite=1, numTasksRead=1, numNodes=1, storageType="localssd"):

    wf_df = pd.DataFrame(columns=wf_params)

    # Find folders ending with [t1, t2, t3] in the test_folders
    test_folders = glob.glob(f"{tests}/*")
    wf_trial_folders = [folder for folder in test_folders if folder.endswith("t1") or folder.endswith("t2") or folder.endswith("t3")]
    # print(f"Found trial folders: {wf_trial_folders} ")

    store_code = transform_store_code(storageType)
    
    for trial_folder in wf_trial_folders:
        # Find all json files in the trial folder
        # wf_json_files = glob.glob(f"{trial_folder}/*.json")


        datalife_jsons = glob.glob(f"{trial_folder}/*.datalife.json")

        # print(datalife_jsons)

        monitor_timer_stat = {}
        task_name = ''
        task_pid = 0


        for datalife_json in datalife_jsons:
            with open(datalife_json) as f:
                # Get task pid from the filename monitor_timer.pid.datalife.json
                datalife_data = json.load(f)
                # Get the first key
                # TODO: get one task now
                task_name = list(datalife_data.keys())[0]
                task_pid = int(datalife_json.split("/")[-1].split(".")[1])
                taskNamePID = f"{task_name}-{task_pid}"

                # check if the task is in the target_tasks, or part of the target_tasks substring
                if any(task in task_name for task in target_tasks):
                    # Select 'monitor', not 'system' and 'local'
                    monitor_timer_stat = datalife_data[task_name]['monitor']
                    break
        # # print(monitor_timer_stat)
        # task_name = datalife_data.keys()[0] # TODO: get one task now

        # Get write statistics
        monitor_timer_stat_write = monitor_timer_stat['write'] # list with [time_sec, op_cnt, io_size]
        print(f"monitor_timer_stat_write = {monitor_timer_stat_write}")
        tmp_write_stat = {}
        tmp_write_stat['aggregateFilesizeMB'] = file_size_to_mb(monitor_timer_stat_write[2])
        if tmp_write_stat['aggregateFilesizeMB'] == 0:
            raise ValueError(f"File size is 0, check the data write, expected write is {monitor_timer_stat_write[2]}")
        tmp_write_stat['numTasks'] = numTasksWrite
        tmp_write_stat['numNodes'] = numNodes
        tmp_write_stat['tasksPerNode'] = tmp_write_stat['numTasks']/tmp_write_stat['numNodes']
        tmp_write_stat['transferSize'] = monitor_timer_stat_write[2]/monitor_timer_stat_write[1]
        tmp_write_stat['operation'] = 0
        tmp_write_stat['totalTime'] = monitor_timer_stat_write[0]
        tmp_write_stat['bwMiB'] = file_size_to_mb(monitor_timer_stat_write[2]/monitor_timer_stat_write[0])
        tmp_write_stat['storageType'] = store_code
        tmp_write_stat['opCount'] = monitor_timer_stat_write[1]
        tmp_write_stat['taskName'] = task_name
        tmp_write_stat['taskPID'] = task_pid

        w_blk_trace_jsons = glob.glob(f"{trial_folder}/*w_blk_trace.json")
        write_pattern = 0 # 0: seq, 1: rand        
        for w_blk_trace_json in w_blk_trace_jsons:
            with open(w_blk_trace_json) as f:
                w_blk_trace_data = json.load(f)
                # print(w_blk_trace_data)
                blk_list = w_blk_trace_data['io_blk_range']
                if not is_sequential(blk_list):
                    write_pattern =1
                    # For now only single read and write
                    break
        tmp_write_stat['randomOffset'] = write_pattern
        wf_df = wf_df._append(tmp_write_stat, ignore_index=True)

        # Get read statistics
        monitor_timer_stat_read = monitor_timer_stat['read']
        tmp_read_stat = {}        
        tmp_read_stat['aggregateFilesizeMB'] = file_size_to_mb(monitor_timer_stat_read[2])
        if tmp_read_stat['aggregateFilesizeMB'] == 0:
            raise ValueError("File size is 0, check the data write")
        tmp_read_stat['numTasks'] = numTasksRead
        tmp_read_stat['numNodes'] = numNodes
        tmp_read_stat['tasksPerNode'] = tmp_read_stat['numTasks']/tmp_read_stat['numNodes']
        tmp_read_stat['transferSize'] = monitor_timer_stat_read[2]/monitor_timer_stat_read[1]
        tmp_read_stat['operation'] = 1
        tmp_read_stat['totalTime'] = monitor_timer_stat_read[0]
        tmp_read_stat['bwMiB'] = file_size_to_mb(monitor_timer_stat_read[2]/monitor_timer_stat_read[0])
        tmp_read_stat['storageType'] = store_code
        tmp_read_stat['opCount'] = monitor_timer_stat_read[1]
        tmp_read_stat['taskName'] = task_name
        tmp_read_stat['taskPID'] = task_pid
        

        r_blk_trace_jsons = glob.glob(f"{trial_folder}/*r_blk_trace.json")
        read_pattern = 0
        for r_blk_trace_json in r_blk_trace_jsons:
            with open(r_blk_trace_json) as f:
                r_blk_trace_data = json.load(f)
                blk_list = r_blk_trace_data['io_blk_range']
                if not is_sequential(blk_list):
                    read_pattern =1
                    # For now only single read and write
                    break
        tmp_read_stat['randomOffset'] = read_pattern
        wf_df = wf_df._append(tmp_read_stat, ignore_index=True)

    # encode string to numbers
    le = preprocessing.LabelEncoder()
    for col in wf_df.columns:
        if pd.api.types.is_string_dtype(wf_df[col]):
            wf_df[col] = le.fit_transform(wf_df[col])

    # print(wf_df.head(5))

    return wf_df

In [422]:
# Load 1kgenome data
onekg_data_path = "./1kgenome_data/1000_tests"


# Key Parameters
wf_params = ['operation', 'randomOffset', 'transferSize', 
            'aggregateFilesizeMB', 'numTasks', 'totalTime', 
            'numNodes', 'tasksPerNode', 'bwMiB', "storageType"]
target_tasks = ["python"] # omit srun from 1kgenome run

all_wf_df = pd.DataFrame(columns=wf_params)

def get_test_folder_dfs(test_folder, wf_params, target_tasks, 
                        storageType="localssd"):
    folder_dfs = pd.DataFrame(columns=wf_params)

    for tests in test_folder:
        # check of test folder starts with seq or par
        if tests.startswith("seq"):
            numTasksWrite = 1
            numTasksRead = 1
        else:
            # get the number after _ps
            num_tasks = int(tests.split("_")[-1].split("ps")[1])
            numTasksWrite = num_tasks
            numTasksRead = num_tasks

        # io_size_dfs
        wf_df = get_wf_result_df(f"{onekg_data_path}/{tests}", wf_params, target_tasks, 
                                numTasksWrite=numTasksWrite, numTasksRead=numTasksRead, 
                                storageType=storageType)
        # print(wf_df.head(5))
        # # Print size of df
        # print(f"df shape: {wf_df.shape}")

        # corr_matrix(wf_df, storageType)
        folder_dfs = folder_dfs._append(wf_df, ignore_index=True)
    return folder_dfs

wf_ssd_df = pd.DataFrame(columns=wf_params)
seq_test_folder = ['seq_1000_1n_ssd_ps1', 'seq_1000_1n_ssd_ps2', 
                'seq_1000_1n_ssd_ps20','seq_1000_1n_ssd_ps40']

wf_ssd_df = wf_ssd_df._append(get_test_folder_dfs(seq_test_folder, 
                                        wf_params, target_tasks, 
                                        storageType="localssd"), ignore_index=True)



par_test_folder = ['par_1000_1n_ssd_ps1','par_1000_1n_ssd_ps2',
                'par_1000_1n_ssd_ps20','par_1000_1n_ssd_ps40']
wf_ssd_df = wf_ssd_df._append(get_test_folder_dfs(par_test_folder,
                                        wf_params, target_tasks, 
                                        storageType="localssd"), ignore_index=True)


wf_beegfs_df = pd.DataFrame(columns=wf_params)
seq_pfs_test_folder = ['seq_1000_1n_pfs_ps1', 'seq_1000_1n_pfs_ps2',
                       'seq_1000_1n_pfs_ps20','seq_1000_1n_pfs_ps40']
wf_beegfs_df = wf_beegfs_df._append(get_test_folder_dfs(seq_pfs_test_folder,
                                        wf_params, target_tasks, 
                                        storageType="beegfs"), ignore_index=True)


par_test_folder = ['par_1000_1n_pfs_ps1','par_1000_1n_pfs_ps2',
                'par_1000_1n_pfs_ps20','par_1000_1n_pfs_ps40']
wf_beegfs_df = wf_beegfs_df._append(get_test_folder_dfs(par_test_folder,
                                        wf_params, target_tasks, 
                                        storageType="beegfs"), ignore_index=True)






monitor_timer_stat_write = [4.504e-05, 4, 31093]
monitor_timer_stat_write = [4.3199e-05, 4, 31088]
monitor_timer_stat_write = [5.5412e-05, 4, 31231]
monitor_timer_stat_write = [6.3702e-05, 4, 31141]


/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/1417136407.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_df = wf_df._append(tmp_write_stat, ignore_index=True)


monitor_timer_stat_write = [5.0692e-05, 4, 31245]
monitor_timer_stat_write = [4.9602e-05, 4, 31146]
monitor_timer_stat_write = [5.6833e-05, 4, 31237]
monitor_timer_stat_write = [3.8791e-05, 4, 31053]
monitor_timer_stat_write = [4.6989e-05, 4, 31375]
monitor_timer_stat_write = [3.7381e-05, 4, 31167]


/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/2347908443.py:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  folder_dfs = folder_dfs._append(wf_df, ignore_index=True)
/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/1417136407.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_df = wf_df._append(tmp_write_stat, ignore_index=True)


monitor_timer_stat_write = [7.0161e-05, 4, 31090]
monitor_timer_stat_write = [3.9041e-05, 4, 52806]
monitor_timer_stat_write = [0.02226697, 4, 31258]
monitor_timer_stat_write = [7.7112e-05, 4, 61829]
monitor_timer_stat_write = [4.5859e-05, 4, 31140]
monitor_timer_stat_write = [7.0262e-05, 4, 47567]
monitor_timer_stat_write = [0.026341332, 4, 31234]
monitor_timer_stat_write = [5.217e-05, 4, 31225]
monitor_timer_stat_write = [6.7621e-05, 4, 52583]
monitor_timer_stat_write = [6.4523e-05, 4, 56853]
monitor_timer_stat_write = [4.728e-05, 4, 31057]
monitor_timer_stat_write = [5.03e-05, 4, 46349]
monitor_timer_stat_write = [0.000100265, 4, 62196]


...

/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/1417136407.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_df = wf_df._append(tmp_write_stat, ignore_index=True)
/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/1417136407.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_df = wf_df._append(tmp_write_stat, ignore_index=True)


monitor_timer_stat_write = [1.613e-05, 1, 30735]
monitor_timer_stat_write = [1.771e-05, 1, 30378]
monitor_timer_stat_write = [1.1501e-05, 1, 47594]
monitor_timer_stat_write = [1.8791e-05, 1, 30436]
monitor_timer_stat_write = [1.6481e-05, 1, 30557]
monitor_timer_stat_write = [1.766e-05, 1, 30757]
monitor_timer_stat_write = [1.681e-05, 1, 30801]
monitor_timer_stat_write = [1.435e-05, 1, 30427]
monitor_timer_stat_write = [1.114e-05, 1, 30899]
monitor_timer_stat_write = [1.6121e-05, 1, 30429]
monitor_timer_stat_write = [1.209e-05, 1, 42219]
monitor_timer_stat_write = [1.022e-05, 1, 30420]
monitor_timer_stat_write = [1.278e-05, 1, 30389]


/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/1417136407.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_df = wf_df._append(tmp_write_stat, ignore_index=True)
/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/2347908443.py:60: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_beegfs_df = wf_beegfs_df._append(get_test_folder_dfs(seq_pfs_test_folder,
/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/1417136407.py:83: FutureWarning: The behavi

monitor_timer_stat_write = [1.238e-05, 1, 50439]
monitor_timer_stat_write = [1.6909e-05, 1, 30415]
monitor_timer_stat_write = [1.45e-05, 1, 44261]


/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/1417136407.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_df = wf_df._append(tmp_write_stat, ignore_index=True)


In [423]:
# # Select df where operation is write
# wf_ssd_write_df = wf_ssd_df[wf_ssd_df['operation'] == 0]
# wf_ssd_read_df = wf_ssd_df[wf_ssd_df['operation'] == 1]
# print(f"wf_ssd_write_df.shape: {wf_ssd_write_df.shape}")
# print(f"wf_ssd_read_df.shape: {wf_ssd_read_df.shape}")

# wf_beegfs_write_df = wf_beegfs_df[wf_beegfs_df['operation'] == 0]
# wf_beegfs_read_df = wf_beegfs_df[wf_beegfs_df['operation'] == 1]
# print(f"wf_beegfs_write_df.shape: {wf_beegfs_write_df.shape}")
# print(f"wf_beegfs_read_df.shape: {wf_beegfs_read_df.shape}")

all_wf_df = merge_by_test_param(wf_ssd_df, wf_beegfs_df)

all_wf_df.to_csv("1000_1n_ssd_pfs.csv")

# Select df where operation is write
all_wf_write_df = all_wf_df[all_wf_df['operation'] == 0]
all_wf_read_df = all_wf_df[all_wf_df['operation'] == 1]
print(f"all_wf_write_df.shape: {all_wf_write_df.shape}")
print(f"all_wf_read_df.shape: {all_wf_read_df.shape}")


allColumns = all_wf_df.columns

# scaled_wf_ssd_df = my_data_transform(wf_ssd_df, cols_to_norm=['aggregateFilesizeMB', 'bwMiB'])
# scaled_wf_beegfs_df = my_data_transform(wf_beegfs_df, cols_to_norm=['aggregateFilesizeMB', 'bwMiB'])

# scaled_wf_df = merge_by_test_param(scaled_wf_ssd_df, scaled_wf_beegfs_df)




Unique 'operation' values in ssd_df: [0 1]
Unique 'operation' values in beegfs_df: [0 1]
Unique 'operation' values in merged_df: [0 1]
  operation randomOffset  transferSize  aggregateFilesizeMB numTasks  \
0         0            1       7755.50             0.029585        2   
1         0            1       7763.25             0.029614        1   
2         0            1       7764.25             0.029618        1   
3         0            1       7767.75             0.029632        1   
4         0            1       7770.75             0.029643        1   

   totalTime_ssd numNodes  tasksPerNode   bwMiB_ssd  opCount  taskName_ssd  \
0       0.000063        1           2.0  466.550253      4.0           0.0   
1       0.000039        1           1.0  763.436069      4.0           0.0   
2       0.000047        1           1.0  626.443808      4.0           0.0   
3       0.000058        1           1.0  511.427789      4.0           0.0   
4       0.000056        1           1.0  5

In [424]:
allColumns = all_wf_df.columns
all_wf_df = all_wf_df.drop_duplicates(subset=allColumns)

# Show 5 rows with selectStorage = 0
print(f"Rows with selectStorage = {decode_store_code(0)}")
print(all_wf_df[all_wf_df['selectStorage'] == 0].head(5))

# Show 5 rows with selectStorage = 1
print(f"Rows with selectStorage = {decode_store_code(1)}")
print(all_wf_df[all_wf_df['selectStorage'] == 1].head(5))

print("Shape of master_ior_df: ", all_wf_df.shape)

all_wf_df.to_csv("1000_1n_ssd_pfs.csv")

Rows with selectStorage = localssd
   operation randomOffset  transferSize  aggregateFilesizeMB numTasks  \
59         1            1    8191.91228          2421.833442        1   
60         1            1    8191.91228          2421.833442        1   
61         1            1    8191.91228          2421.833442        1   
62         1            1    8191.91228          2421.833442        1   
63         1            1    8191.91228          2421.833442        1   

    totalTime_ssd numNodes  tasksPerNode    bwMiB_ssd   opCount  taskName_ssd  \
59       0.533575        1           1.0  4538.877356  309998.0           0.0   
60       0.533575        1           1.0  4538.877356  309998.0           0.0   
61       0.533575        1           1.0  4538.877356  309998.0           0.0   
62       0.533575        1           1.0  4538.877356  309998.0           0.0   
63       0.533575        1           1.0  4538.877356  309998.0           0.0   

    taskPID_ssd  totalTime_beegfs  bwMi

In [425]:
# TODO: Use a linear interpolation function to estimate the bandwidth for each row in scaled_wf_df

# Read from file "./master_ior_df.csv"
df_ior = pd.read_csv("./master_ior_df.csv")
# oscache size is 25GiB
oscacheSizeMB = 25 * 1024  # Convert to MiB

# Function to calculate bandwidth based on bounds
def calculate_bandwidth_storage(df_ior, tasks_per_node, transfer_size, bandwidth_column='bwMiB_ave_ssd'):
    # Sort df_ior by tasksPerNode and transferSize
    df_ior_sorted = df_ior.sort_values(by=['tasksPerNode', 'transferSize'])
    
    # Find lower and upper bounds for tasksPerNode
    lower_tasks = df_ior_sorted[df_ior_sorted['tasksPerNode'] <= tasks_per_node]
    upper_tasks = df_ior_sorted[df_ior_sorted['tasksPerNode'] >= tasks_per_node]
    
    if lower_tasks.empty or upper_tasks.empty:
        raise ValueError("No rows found within tasksPerNode bounds.")
    
    low_bound_tasks = lower_tasks.iloc[-1]
    high_bound_tasks = upper_tasks.iloc[0]
    
    # Filter df_ior by tasksPerNode bounds
    df_ior_within_bounds = df_ior_sorted[
        (df_ior_sorted['tasksPerNode'] >= low_bound_tasks['tasksPerNode']) &
        (df_ior_sorted['tasksPerNode'] <= high_bound_tasks['tasksPerNode'])
    ]
    
    if df_ior_within_bounds.empty:
        raise ValueError("No rows found within tasksPerNode bounds.")
    
    # Sort within bounds by transferSize
    df_ior_within_bounds = df_ior_within_bounds.sort_values(by='transferSize')
    
    # Check if there are enough rows to find transferSize bounds
    low_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] < transfer_size]
    high_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] > transfer_size]
    
    if low_bound_transfer.empty or high_bound_transfer.empty:
        raise ValueError("Not enough data to determine bounds within transferSize bounds.")
    
    low_bound = low_bound_transfer.iloc[-1]
    high_bound = high_bound_transfer.iloc[0]
    
    # Perform linear interpolation to estimate the bandwidth
    low_size = low_bound['transferSize']
    high_size = high_bound['transferSize']
    low_bw = low_bound[bandwidth_column]
    high_bw = high_bound[bandwidth_column]
    
    estimated_bw = low_bw + (transfer_size - low_size) * (high_bw - low_bw) / (high_size - low_size)
    return estimated_bw

# Calculate the theoretical memory bandwidth for current task
def calculate_bandwidth_memory(df_ior, operation, tasks_per_node, transfer_size, rw_ratio_column='RWR_ssd'):
    # Sort df_ior by tasksPerNode and transferSize
    df_ior_sorted = df_ior.sort_values(by=['tasksPerNode', 'transferSize'])
    
    # Find lower and upper bounds for tasksPerNode
    lower_tasks = df_ior_sorted[df_ior_sorted['tasksPerNode'] <= tasks_per_node]
    upper_tasks = df_ior_sorted[df_ior_sorted['tasksPerNode'] >= tasks_per_node]
    
    if lower_tasks.empty or upper_tasks.empty:
        raise ValueError("No rows found within tasksPerNode bounds.")
    
    low_bound_tasks = lower_tasks.iloc[-1]
    high_bound_tasks = upper_tasks.iloc[0]
    
    # Filter df_ior by tasksPerNode bounds
    df_ior_within_bounds = df_ior_sorted[
        (df_ior_sorted['tasksPerNode'] >= low_bound_tasks['tasksPerNode']) &
        (df_ior_sorted['tasksPerNode'] <= high_bound_tasks['tasksPerNode'])
    ]
    
    if df_ior_within_bounds.empty:
        raise ValueError("No rows found within tasksPerNode bounds.")
    
    # Sort within bounds by transferSize
    df_ior_within_bounds = df_ior_within_bounds.sort_values(by='transferSize')
    
    # Check if there are enough rows to find transferSize bounds
    low_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] < transfer_size]
    high_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] > transfer_size]
    
    if low_bound_transfer.empty or high_bound_transfer.empty:
        raise ValueError("Not enough data to determine bounds within transferSize bounds.")
    
    low_bound = low_bound_transfer.iloc[-1]
    high_bound = high_bound_transfer.iloc[0]
    
    # Perform linear interpolation to estimate the bandwidth
    low_size = low_bound['transferSize']
    high_size = high_bound['transferSize']
    low_rwr = low_bound[rw_ratio_column]
    high_rwr = high_bound[rw_ratio_column]
    
    estimated_rwr = low_rwr + (transfer_size - low_size) * (high_rwr - low_rwr) / (high_size - low_size)
    
    # DDR4-2400	DDR4-2400 Peak Transfer Rate 19.2GB/s
    # Maximum Memory Speed 2400 MHz	
    # Max # of Memory Channels: 6
    # Total Bandwidth=19.2GB/s * 6 = 115.2GB/s
    # convert to MiB/s
    theoretical_read_bw_mbs = 115.2 * 1024
    contention_slowdown = 0.8 # FIXME: find a better value
    
    if tasks_per_node <= 6:
        if operation == 0:
            estimated_bw = theoretical_read_bw_mbs / estimated_rwr
        else:
            estimated_bw = theoretical_read_bw_mbs
    else:
        exccedd_tasks = tasks_per_node - 6
        if operation == 0:
            estimated_bw = (contention_slowdown * exccedd_tasks) * theoretical_read_bw_mbs / estimated_rwr
        else:
            estimated_bw = (contention_slowdown * exccedd_tasks) * theoretical_read_bw_mbs
    
    # Check if estimated_bw is nan, then print other values
    if np.isnan(estimated_bw):
        # Default to theoretical_read_bw_mbs
        estimated_bw = theoretical_read_bw_mbs
        # print(f"Estimated BW: {estimated_bw}, estimated_rwr: {estimated_rwr}, theoretical_read_bw_mbs: {theoretical_read_bw_mbs}")
    
    return estimated_bw
    

# Iterate through each row of scaled_wf_df
for index, row in all_wf_df.iterrows():
    # Find matching rows in df_ior based on operation
    conditions = (
        (df_ior['operation'] == row['operation'])
    ) 
    df_ior_matched = df_ior[conditions]

    if not df_ior_matched.empty:
        # Find bounds for the transferSize
        if len(df_ior_matched) > 1:
            try:
                aggregateFilesizeMB = row['aggregateFilesizeMB']
                
                # Calculate estimated bandwidth
                estimated_bwMiB_ssd = calculate_bandwidth_storage(df_ior_matched, row['tasksPerNode'], row['transferSize'],bandwidth_column='bwMiB_ave_ssd')
                all_wf_df.at[index, 'estimated_bwMiB_ssd'] = estimated_bwMiB_ssd
                estimated_bwMiB_beegfs = calculate_bandwidth_storage(df_ior_matched, row['tasksPerNode'], row['transferSize'],bandwidth_column='bwMiB_ave_beegfs')
                all_wf_df.at[index, 'estimated_bwMiB_beegfs'] = estimated_bwMiB_beegfs

            except ValueError as e:
                print(f"Row {index}: Error calculating bandwidth - {e}")
                all_wf_df.at[index, 'estimated_bw'] = 0
        else:
            print(f"Row {index}: Not enough data to determine bounds.")
            all_wf_df.at[index, 'estimated_bw'] = 0
    else:
        print(f"Row {index}: No matching rows in df_ior.")
        all_wf_df.at[index, 'estimated_bw'] = 0
        # print(f" - Row {row.name}: TransferSize {row['transferSize']} kb")
        # print(f" - Row {row.name}: operation {row['operation']}, randomOffset {row['randomOffset']}, tasksPerNode {row['tasksPerNode']}")



In [426]:
# Check df with operation = 0 and 1
# Select df where operation is write
all_wf_write_df = all_wf_df[all_wf_df['operation'] == 0]
print(f"all_wf_write_df.shape: {all_wf_write_df.shape}")
all_wf_read_df = all_wf_df[all_wf_df['operation'] == 1]
print(f"all_wf_read_df.shape: {all_wf_read_df.shape}")

all_wf_write_df.shape: (59, 19)
all_wf_read_df.shape: (348, 19)


In [427]:
# TODO: Add a expected memory access time to get the expected best bwMiB
# A dictionary with expected file size limit and the expected memory access performance, then time
# DDR4-2400	DDR4-2400 Peak Transfer Rate 19.2GB/s
# Maximum Memory Speed 2400 MHz	
# Max # of Memory Channels: 6
# Total Bandwidth=19.2GB/s * 6 = 115.2GB/s
# https://docs.google.com/spreadsheets/d/1eHgRg3jEjf00q9l6CXmOAyeirnDOXOlM2DUrSAJvKTc/edit?gid=1374170985#gid=1374170985

mem_access_dictMB = {
    # 25GB expressed in MB size, then {IO_size: [bwGB/s, readTimeSec, writeTimeSec] } 
    25600: {
        # 64 bytes read is 2.6 times write performance speed
        64: [115.2, 0.222, 0.085], 
        # 1KB read is 4.5 times write performance speed
        1024: [115.2, 0.222, 0.049],
        # 4KB read is 7 times write performance speed
        4096: [115.2, 0.222, 0.032],
        # 1MB read is 10 times write performance speed
        1048576: [115.2, 0.222, 0.022],
        # 4MB read is 14 times write performance speed
        4194304: [115.2, 0.222, 0.016],
        # 64MB read is 8 times write performance speed
        67108864: [115.2, 0.222, 0.028],
        # 1GB read is 7 times write performance speed
        1073741824: [115.2, 0.222, 0.032],
    },
    # 256GB expressed in MB size, then {IO_size: [bwGB/s, readTimeSec, writeTimeSec] }
    262144: {
        "localssd":{
            # 1KB read is 4 times write performance speed
            1024: [115.2, 0.222, 0.055],
            # 4KB read is 8 times write performance speed
            4096: [115.2, 0.222, 0.028],
            # 1MB read is 11 times write performance speed
            1048576: [115.2, 0.222, 0.020],
            # 4MB read is 15 times write performance speed
            4194304: [115.2, 0.222, 0.015],
            # 64MB read is 7 times write performance speed
            67108864: [115.2, 0.222, 0.032],
            # 1GB read is 7 times write performance speed
            1073741824: [115.2, 0.222, 0.032],
        },
        "pfs":{            
            # 1KB read is 4 times write performance speed
            1024: [115.2, 0.222, 0.055],
            # 4KB read is 8 times write performance speed
            4096: [115.2, 0.222, 0.028],
            # 1MB read is 11 times write performance speed
            1048576: [115.2, 0.222, 0.020],
            # 4MB read is 15 times write performance speed
            4194304: [115.2, 0.222, 0.015],
            # 64MB read is 7 times write performance speed
            67108864: [115.2, 0.222, 0.032],
            # 1GB read is 7 times write performance speed
            1073741824: [115.2, 0.222, 0.032],
        }

    }
}


In [428]:
# print(scaled_wf_df.head(5))
all_wf_df.to_csv("1000_1n_ssd_pfs.csv")
# print(scaled_wf_df.head(5))
print(all_wf_df.shape)

# Show row when estimated_bwMiB_beegfs has value larger than estimated_bwMiB_ssd
srows1 = all_wf_df[all_wf_df['estimated_bwMiB_beegfs'] > all_wf_df['estimated_bwMiB_ssd']]
print(srows1.shape)

# Show row when estimated_bwMiB_beegfs has value smaller than estimated_bwMiB_ssd
srows2 = all_wf_df[all_wf_df['estimated_bwMiB_beegfs'] < all_wf_df['estimated_bwMiB_ssd']]
print(srows2.shape)


(407, 19)
(53, 19)
(354, 19)


In [429]:
# # Select storageType
# wf_ssd_df = all_wf_df[all_wf_df['storageType'] == 0]
# print(f"SSD DF: {wf_ssd_df.shape}")
# wf_beegfs_df = all_wf_df[all_wf_df['storageType'] == 1]
# print(f"BeeGFS DF: {wf_beegfs_df.shape}")


def calculate_taskbw(df, suffix):
    # Get unique transfer sizes and number of tasks
    numTasks = sorted(df['numTasks'].unique())
    task_pids = sorted(df[f'taskPID_{suffix}'].unique())
    print(f"numTasks: {numTasks}, task_pids: {task_pids}")
    
    # Initialize the 'task_bwMiB_{suffix}' column if it doesn't exist
    if f'task_bwMiB_{suffix}' not in df.columns:
        df[f'task_bwMiB_{suffix}'] = 0.0
    
    for ntasks in numTasks:
        for pid in task_pids:
            filtered_df = df[(df['numTasks'] == ntasks) & (df[f'taskPID_{suffix}'] == pid)]

            if not filtered_df.empty:
                # Find the unique pairs of 'operation' and 'opCount'
                df_select = filtered_df[['operation', 'opCount', f'bwMiB_{suffix}']].drop_duplicates()
                
                # Convert to dictionary with { operation: [(opCount, bwMiB_{suffix}), ...] }
                op_count_dict = df_select.groupby('operation').apply(
                    lambda x: list(zip(x['opCount'], x[f'bwMiB_{suffix}']))
                    ,include_groups=False
                ).to_dict()

                # print(f"{suffix} op_count_dict: {op_count_dict}")
                
                # Calculate the new column task_estimated_bwMiB_{suffix} with operation count weight
                if len(op_count_dict) > 0:
                    # Flatten the list of tuples for all operations present
                    total_op_count = sum(opCount for opCount, _ in sum(op_count_dict.values(), []))
                    # Calculate proportions and weighted bandwidth
                    write_op_bw = read_op_bw = 0
                    for operation, values in op_count_dict.items():
                        op_count_sum = sum(opCount for opCount, _ in values)
                        op_portion = op_count_sum / total_op_count
                        bw_sum = sum(bwMiB * op_portion for _, bwMiB in values)
                        
                        if operation == 0: write_op_bw = bw_sum
                        elif operation == 1: read_op_bw = bw_sum
                    # Estimated bandwidth
                    task_bwMiB_storage = write_op_bw + read_op_bw
                    # print(f"Task Bandwidth: {task_bwMiB_storage}")
                    
                    # Update the original DataFrame with the new estimated bandwidth
                    df.loc[(df['numTasks'] == ntasks) & (df[f'taskPID_{suffix}'] == pid), f'task_bwMiB_{suffix}'] = task_bwMiB_storage
                    
                else:
                    print("No operations available for calculation.")

                    
    return df


# all_wf_write_df = all_wf_df[all_wf_df['operation'] == 0]
# print(f"all_wf_write_df.shape: {all_wf_write_df.shape}")

# print all columns
print(all_wf_df.columns)

# Fill the empty taskName_ssd values with -1
all_wf_df['taskName_ssd'] = all_wf_df['taskName_ssd'].fillna(-1)
all_wf_df['taskName_beegfs'] = all_wf_df['taskName_beegfs'].fillna(-1)





all_wf_df = all_wf_df.groupby('taskName_ssd').apply(calculate_taskbw, suffix='ssd', include_groups=True).reset_index(drop=True)



all_wf_df = all_wf_df.groupby('taskName_beegfs').apply(calculate_taskbw, suffix='beegfs', include_groups=True).reset_index(drop=True)






Index(['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMB',
       'numTasks', 'totalTime_ssd', 'numNodes', 'tasksPerNode', 'bwMiB_ssd',
       'opCount', 'taskName_ssd', 'taskPID_ssd', 'totalTime_beegfs',
       'bwMiB_beegfs', 'taskName_beegfs', 'taskPID_beegfs', 'selectStorage',
       'estimated_bwMiB_ssd', 'estimated_bwMiB_beegfs'],
      dtype='object')
numTasks: [1, 2, 20, 40], task_pids: [np.float64(nan)]
numTasks: [1, 2, 20, 40], task_pids: [np.float64(27590.0), np.float64(28507.0), np.float64(47903.0), np.float64(48860.0), np.float64(49836.0), np.float64(51071.0), np.float64(52528.0), np.float64(53990.0), np.float64(55748.0), np.float64(57617.0), np.float64(59669.0), np.float64(71708.0), np.float64(72713.0), np.float64(73734.0), np.float64(74758.0), np.float64(75906.0), np.float64(77442.0), np.float64(77451.0), np.float64(78937.0), np.float64(80555.0), np.float64(82656.0), np.float64(84850.0), np.float64(104050.0), np.float64(104995.0), np.float64(105925.0), np

/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/2141825201.py:75: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  all_wf_df = all_wf_df.groupby('taskName_ssd').apply(calculate_taskbw, suffix='ssd', include_groups=True).reset_index(drop=True)
/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/2141825201.py:79: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  all_wf_df = all_wf_df.gr

In [430]:
def calculate_ebw(df, suffix):
    # Get unique transfer sizes and number of tasks
    numTasks = sorted(df['numTasks'].unique())
    task_pids = sorted(df[f'taskPID_{suffix}'].unique())

    # Initialize the 'task_bwMiB_{suffix}' column if it doesn't exist
    if f'task_estimated_bwMiB_{suffix}' not in df.columns:
        df[f'task_estimated_bwMiB_{suffix}'] = 0.0
        
    for ntasks in numTasks:
        for pid in task_pids:
            filtered_df = df[(df['numTasks'] == ntasks) & (df[f'taskPID_{suffix}'] == pid)]

            if not filtered_df.empty:
                # Find the unique pairs of 'operation' and 'opCount'
                df_select = filtered_df[['operation', 'opCount', f'estimated_bwMiB_{suffix}']].drop_duplicates()
                
                # Convert to dictionary with { operation: [(opCount, estimated_bwMiB_{suffix}), ...] }
                op_count_dict = df_select.groupby('operation').apply(
                    lambda x: list(zip(x['opCount'], x[f'estimated_bwMiB_{suffix}']))
                    ,include_groups=False
                ).to_dict()

                # print(f"{suffix} op_count_dict: {op_count_dict}")
                
                # Calculate the new column task_estimated_bwMiB_{suffix} with operation count weight
                if len(op_count_dict) > 0:
                    # Flatten the list of tuples for all operations present
                    total_op_count = sum(opCount for opCount, _ in sum(op_count_dict.values(), []))

                    if total_op_count > 0:
                        # Initialize write and read weights to 0
                        op_write_weight = op_read_weight = 0

                        # Calculate proportions and weighted bandwidth
                        write_op_bw = read_op_bw = 0
                        for operation, values in op_count_dict.items():
                            op_count_sum = sum(opCount for opCount, _ in values)
                            op_portion = op_count_sum / total_op_count
                            
                            # Calculate the bandwidth sum weighted by the operation portion
                            bw_sum = sum(bwMiB * op_portion for _, bwMiB in values)

                            # Assign weights and bandwidth for write and read operations
                            if operation == 0:  # Write operation
                                op_write_weight = op_portion
                                write_op_bw = bw_sum
                            elif operation == 1:  # Read operation
                                op_read_weight = op_portion
                                read_op_bw = bw_sum

                        # Calculate the total estimated bandwidth
                        estimated_bwMiB_storage = (write_op_bw * op_write_weight) + (read_op_bw * op_read_weight)
                        # print(f"Write weight: {op_write_weight}, Read weight: {op_read_weight}, Estimated Bandwidth: {estimated_bwMiB_storage}")
                    
                        # Update the original DataFrame with the new estimated bandwidth
                        df.loc[(df['numTasks'] == ntasks) & (df[f'taskPID_{suffix}'] == pid), f'task_estimated_bwMiB_{suffix}'] = estimated_bwMiB_storage
                    else:
                        print(f"No operation counts available for ntasks={ntasks} and pid={pid}.")
                else:
                    print(f"No operations available for calculation for ntasks={ntasks} and pid={pid}.")
                    
    return df

all_wf_df = all_wf_df.groupby('taskName_ssd').apply(calculate_ebw, suffix='ssd', include_groups=True).reset_index(drop=True)
all_wf_df = all_wf_df.groupby('taskName_beegfs').apply(calculate_ebw, suffix='beegfs', include_groups=True).reset_index(drop=True)

# Create a new column of estimated_task_selectStorage comparing the estimated bandwidths of ssd and beegfs
all_wf_df['task_selectStorage'] = all_wf_df.apply(
    lambda row: 0 if row['task_bwMiB_ssd'] > row['task_bwMiB_beegfs'] else 1, axis=1
)
all_wf_df['estimated_task_selectStorage'] = all_wf_df.apply(
    lambda row: 0 if row['task_estimated_bwMiB_ssd'] > row['task_estimated_bwMiB_beegfs'] else 1, axis=1
)


# print(grouped_df.head(5))


# Save all_wf_df
all_wf_df.to_csv("1000_1n_ssd_pfs_last.csv")


/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/4090332646.py:65: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  all_wf_df = all_wf_df.groupby('taskName_ssd').apply(calculate_ebw, suffix='ssd', include_groups=True).reset_index(drop=True)
/var/folders/t1/j5nks1f97pzdzx63f2_4_65m0000gn/T/ipykernel_18912/4090332646.py:66: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  all_wf_df = all_wf_df.group

In [431]:
# Check if task_selectStorage and estimated_task_selectStorage is the same, print the rows that are different
diff_rows = all_wf_df[all_wf_df['task_selectStorage'] != all_wf_df['estimated_task_selectStorage']]
if diff_rows.empty:
    print("All rows from task_selectStorage and estimated_task_selectStorage are the same.")
else:
    print("Rows where task_selectStorage and estimated_task_selectStorage are different:")
    print(diff_rows)



All rows from task_selectStorage and estimated_task_selectStorage are the same.


In [432]:
# Check the variance between bwMiB_ssd and estimated_bwMiB_ssd
all_wf_df['bwMiB_ssd_variance'] = all_wf_df['bwMiB_ssd'] - all_wf_df['estimated_bwMiB_ssd']
all_wf_df['bwMiB_beegfs_variance'] = all_wf_df['bwMiB_beegfs'] - all_wf_df['estimated_bwMiB_beegfs']

average_percent_variance_ssd = abs(all_wf_df['bwMiB_ssd_variance'].mean()/all_wf_df['bwMiB_ssd'].mean())
average_percent_variance_beegfs = abs(all_wf_df['bwMiB_beegfs_variance'].mean()/all_wf_df['bwMiB_beegfs'].mean())

print(f"Average percent variance for ssd: {average_percent_variance_ssd}")
print(f"Average percent variance for beegfs: {average_percent_variance_beegfs}")

Average percent variance for ssd: 0.8087857170295021
Average percent variance for beegfs: 0.6451022431776116


In [433]:
# Check the variance between task_bwMiB_ssd and task_estimated_bwMiB_ssd
all_wf_df['task_bwMiB_ssd_variance'] = all_wf_df['task_bwMiB_ssd'] - all_wf_df['task_estimated_bwMiB_ssd']
all_wf_df['task_bwMiB_beegfs_variance'] = all_wf_df['task_bwMiB_beegfs'] - all_wf_df['task_estimated_bwMiB_beegfs']

average_percent_variance_task_ssd = abs(all_wf_df['task_bwMiB_ssd_variance'].mean()/all_wf_df['task_bwMiB_ssd'].mean())
average_percent_variance_task_beegfs = abs(all_wf_df['task_bwMiB_beegfs_variance'].mean()/all_wf_df['task_bwMiB_beegfs'].mean())

print(f"Average percent variance for task ssd: {average_percent_variance_task_ssd}")
print(f"Average percent variance for task beegfs: {average_percent_variance_task_beegfs}")


Average percent variance for task ssd: 1.0009565304838668
Average percent variance for task beegfs: 1.0009713038611863
